## Import necessary modules
Run this cell before running any other cells

In [1]:
 pip install numpy pyyaml colorama nest_asyncio bleak jupyterlab

Note: you may need to restart the kernel to use updated packages.


In [1]:
#%load_ext autoreload
#%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

# Printing and Logging
## Printing
You can use the **print()** function in Python to print messages to the screen. <br>
The message can be a string, or any other object, the object will be converted into a string before it is written to the screen. <br>

## Logging
You could use the logging module that is setup in *utils.py*. <br>
It prints to both your screen (standard output) as well as to log files (*ble.log*) in the *logs* directory. <br>
This is the recommended way to output messages, since the log files can help with debugging. <br>
The logging module also provides different log levels as shown below, each formatted with a different color for increased visibility. <br>

__**NOTE**__: You may notice that the DEBUG message is not printed to the screen but is printed in the log file. This is because the logging level for the screen is set to INFO and for the file is set to DEBUG. You can change the default log levels in *utils.py* (**STREAM_LOG_LEVEL** and **FILE_LOG_LEVEL**). 

## Formatting output
To format your strings, you may use %-formatting, str.format() or f-strings. <br>
The most "pythonic" way would be to use f-strings. [Here](https://realpython.com/python-f-strings/) is a good tutorial on f-strings. <br>

In [2]:
LOG.debug("debug")
LOG.info("info")
LOG.warning("warning")
LOG.error("error")
LOG.critical("critical")

2026-02-03 00:14:01,464 | INFO     |: info
2026-02-03 00:14:01,465 | WARNING  |: warning
2026-02-03 00:14:01,466 | ERROR    |: error
2026-02-03 00:14:01,467 | CRITICAL |: critical


<hr>

# BLE
## ArtemisBLEController
The class **ArtemisBLEController** (defined in *ble.py*) provides member functions to handle various BLE operations to send and receive data to/from the Artemis board, provided the accompanying Arduino sketch is running on the Artemis board. <br>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Functions</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">reload_config()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Reload changes made in <em>connection.yaml.</em></span></th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">connect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Connect to the Artemis board, whose MAC address is specified in <em>connection.yaml</em>.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">disconnect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Disconnect from the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">is_connected()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Return a boolean indicating whether your controller is connected to the Artemis board or not.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">send_command(cmd_type, data)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Send the command <strong>cmd_type</strong> (integer) with <strong>data</strong> (string) to the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">receive_float(uuid) <br> receive_string(uuid) <br> receive_int(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Read the GATT characteristic (specified by its <strong>uuid</strong>) of type float, string or int. <br> The type of the GATT
            characteristic is determined by the classes BLEFloatCharacteristic, BLECStringCharacteristic or
            BLEIntCharacteristic in the Arduino sketch.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">start_notify(uuid, notification_handler)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Activate notifications on the GATT characteristic (specified by its <strong>uuid</strong>). <br> <strong>notification_handler</strong> is a
            function callback which must accept two inputs; the first will be a uuid string object and the second will
            be the bytearray of the characteristic value.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">bytearray_to_float(byte_array) <br> bytearray_to_string(byte_array) <br> bytearray_to_int(byte_array)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Convert the <strong>bytearray</strong> to float, string or int, respectively. <br> You may use these functions inside your
            notification callback function.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">stop_notify(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Stop notifications on the GATT characteristic (specified by its <strong>uuid</strong>).</span></th>
    </tr>
</table>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Variables</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">uuid</span></th>
        <th style="text-align: left"><span style="font-weight: normal">A dictionary that stores the UUIDs of the various characteristics specified in <em>connection.yaml</em>.</span></th>
    </tr>
</table>

## Configuration
- The MAC address, Service UUID and GATT characteristic UUIDs are defined in the file: *connection.yaml*.
- They should match the UUIDs used in the Arduino sketch.
- The artemis board running the base code should display its MAC address in the serial monitor.
- Update the **artemis_address** in *connection.yaml*, accordingly.
- Make sure to call **ble.reload_config()** or **get_ble_controller()** (which internally calls **reload_config()**) after making any changes to your configuration file.

<hr>

In the below cell, we create an **ArtemisBLEController** object using **get_ble_controller()** (defined in *ble.py*), which creates and/or returns a single instance of **ArtemisBLEController**. <br>
<span style="color:rgb(240,50,50)"> __NOTE__: Do not use the class directly to instantiate an object. </span><br>

In [3]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

2026-02-03 00:14:04,846 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:81:b4:24:2b:64
2026-02-03 00:14:04,848 | INFO     |: Scanning for device with address: c0:81:b4:24:2b:64, service UUID: 15bb5de7-5941-4ba2-bda0-784bb8817a1b
2026-02-03 00:14:15,052 | INFO     |: Found 1 device(s) advertising service 15bb5de7-5941-4ba2-bda0-784bb8817a1b
2026-02-03 00:14:15,060 | INFO     |: Selecting device: 38645F53-E5BF-2155-7DF0-DBDE5B0B8B54 (name: Artemis BLE)
2026-02-03 00:14:16,006 | INFO     |: Connected to c0:81:b4:24:2b:64


## Receive data from the Artemis board

The cell below shows examples of reading different types (as defined in the Arduino sketch) of GATT characteristics.

In [46]:
# Read a float GATT Charactersistic
f = ble.receive_float(ble.uuid['RX_FLOAT'])
print(f)

12.5


In [45]:
# Read a string GATT Charactersistic
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

[->9.000<-]


## Send a command to the Artemis board
Send the PING command and read the reply string from the string characteristic RX_STRING. <br>
__NOTE__: The **send_command()** essentially sends a string data to the GATT characteristic (TX_CMD_STRING). The GATT characteristic in the Arduino sketch is of type BLECStringCharacteristic.

In [47]:
ble.send_command(CMD.PING, "")

In [48]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

PONG


The cell below shows an example of the SEND_TWO_INTS command. <br> The two values in the **data** are separated by a delimiter "|". <br>
Refer Lab 2 documentation for more information on the command protocol.

In [34]:
ble.send_command(CMD.SEND_TWO_INTS, "2|-6")

The Artemis board should print the two integers to the serial monitor in the ArduinoIDE. 

***Task 1 - ECHO command*** <br>
The computer sends string value (e.g. "HiHello") to Artemis board. <br>
The Artemis board sends back augmented string (e.g. “Robot says -> HiHello :)” from a read GATT characteristic.

In [35]:
ble.send_command(CMD.ECHO, "HiHello")

In [11]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

Robot says -> HiHello :)


***Task 2 - SEND_THREE_FLOATS command*** <br>
Send 3 floats to Artemis board. <br>
Extract the three float values in the Arduino sketch.

In [12]:
ble.send_command(CMD.SEND_THREE_FLOATS, "2.0|-6.0|0.39")

***Task 3 - GET_TIME_MILLIS*** <br>
Robot reply write a string such as "T:123456" to the string characteristic.

In [11]:
ble.send_command(CMD.GET_TIME_MILLIS, "")

78916


In [9]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

T:72216


***Task 4 - Notification Handler*** <br>
Setup notification handler in Python to receive string value from Artemis board. <br>
In callback function, extract the time from string.


In [49]:
times_int = []
times_str = []
def get_time(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    # later for calculating data transfer rate
    times_str.append(s)
    # only take values after colon
    s = s.split(":")[1]
    # convert this string to int
    s = int(s)
    # later for calculating data transfer rate
    times_int.append(s)
    print(s)

In [50]:
ble.start_notify(ble.uuid['RX_STRING'], get_time)

In [51]:
ble.stop_notify(ble.uuid['RX_STRING'])

***Task 5 - Loop to get current time*** <br>
Write a loop to get current time in miliseconds. <br>
Send the current times to laptop to be received and processed by the notification handler. <br>
Collect these values for a few seconds and use time stamps to determine how fast messages can be sent.

In [32]:
ble.start_notify(ble.uuid['RX_STRING'], get_time)

In [33]:
ble.send_command(CMD.GET_CURRRENT_TIME_LOOP, "")

576442
576443
576444
576445
576507
576508
576533
576534
576535
576567
576568
576626
576627
576627
576654
576655
576689
576690
576690
576717
576718
576744
576745
576746
576777
576778
576803
576813
576813
576838
576839
576840
576863
576873
576901
576902
576903
576904
576926
576927
576959
576960
576960
576988
576989
577014
577015
577016
577047
577048
577075
577076
577102
577135
577136
577170
577171
577198
577227
577228
577253
577263
577290
577315
577316
577317
577349
577375
577409
577410
577437
577438
577465
577497
577498
577523
577533
577560
577586
577586
577587
577619
577679
577705
577706
577739
577740
577766
577802
577803
577828
577829
577862
577886
577887
577921
577922
577949
577981
577982


In [34]:
time_elapsed = times_int[-1] - times_int[0]
num_times = len(times_int)
data_transfer_rate_msg_per_ms = num_times/time_elapsed
# get_time receives byte_array, something like T:2665594, which is 9 bytes long
# calculate total bytes in times
total_times_byte = sum(len(time) for time in times_str)
data_transfer_rate_byte_per_s = total_times_byte/time_elapsed * 1000

# print results
print(f"Time elapsed = {time_elapsed}")
print(f"Number of logged time = {num_times}")
print(f"Data transfer rate (msg/ms) = {data_transfer_rate_msg_per_ms}")
print(f"Total number of bytes send = {total_times_byte}")
print(f"Data transfer rate (byte/s) = {data_transfer_rate_byte_per_s}")


Time elapsed = 1540
Number of logged time = 100
Data transfer rate (msg/ms) = 0.06493506493506493
Total number of bytes send = 800
Data transfer rate (byte/s) = 519.4805194805194


In [35]:
ble.stop_notify(ble.uuid['RX_STRING'])

***Task 6*** <br>
SEND_TIME_DATA loops array and sends individual data point as a string to laptop to be processed. <br>

In [36]:
times_int = []
times_str = []

In [37]:
ble.start_notify(ble.uuid['RX_STRING'], get_time)

In [38]:
ble.send_command(CMD.STORE_TIME_STAMP, "")

In [39]:
ble.send_command(CMD.SEND_TIME_DATA, "")

583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583596
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583597
583598
583598
583598


In [40]:
# to check time difference between task 5 (send each data through ble) and task 6 (send to ble after artemis get all data)
time_elapsed = times_int[-1] - times_int[0]
num_times = len(times_int)
data_transfer_rate_msg_per_ms = num_times/time_elapsed
# get_time receives byte_array, something like T:2665594, which is 9 bytes long
# calculate total bytes in times
total_times_byte = sum(len(time) for time in times_str)
data_transfer_rate_byte_per_s = total_times_byte/time_elapsed * 1000

# print results
print(f"Time elapsed = {time_elapsed}")
# check to see  if all data received
print(f"Number of logged time = {num_times}")
print(f"Data transfer rate (msg/ms) = {data_transfer_rate_msg_per_ms}")
print(f"Total number of bytes send = {total_times_byte}")
print(f"Data transfer rate (byte/s) = {data_transfer_rate_byte_per_s}")

Time elapsed = 2
Number of logged time = 100
Data transfer rate (msg/ms) = 50.0
Total number of bytes send = 800
Data transfer rate (byte/s) = 400000.0


In [20]:
ble.stop_notify(ble.uuid['RX_STRING'])

***Task 7*** <br>
Notification handler parse strings and populate the data into two lists.

In [4]:
# notification handler new callback function to get time and temperature reading
temp_float_array = []
time_int_array = []
def get_time_temp(uuid, byte_array):
    s = ble.bytearray_to_string(byte_array)
    # first split temperature and time
    temp_s = s.split(",")[0]
    time_s = s.split(",")[1]
    # then take value after colon and convert to int or float
    temp = float(temp_s.split(":")[1])
    time = int(time_s.split(":")[1])
    # add to array
    temp_float_array.append(temp)
    time_int_array.append(time)
    
    print(f"Temperature:{temp},Time:{time}")

In [5]:
ble.start_notify(ble.uuid['RX_STRING'], get_time_temp)

In [6]:
ble.send_command(CMD.STORE_TIME_TEMP_STAMP, "")

In [7]:
ble.send_command(CMD.GET_TEMP_READINGS, "")

Temperature:77.597,Time:274650
Temperature:76.54,Time:274651
Temperature:76.54,Time:274651
Temperature:76.54,Time:274651
Temperature:76.54,Time:274651
Temperature:76.54,Time:274652
Temperature:76.54,Time:274652
Temperature:76.54,Time:274652
Temperature:77.597,Time:274652
Temperature:74.426,Time:274653
Temperature:76.54,Time:274653
Temperature:77.597,Time:274653
Temperature:76.54,Time:274653
Temperature:76.54,Time:274654
Temperature:75.483,Time:274654
Temperature:76.54,Time:274654
Temperature:76.54,Time:274654
Temperature:76.54,Time:274654
Temperature:76.54,Time:274658
Temperature:76.54,Time:274658
Temperature:76.54,Time:274658
Temperature:77.597,Time:274658
Temperature:75.483,Time:274659
Temperature:76.54,Time:274659
Temperature:76.54,Time:274659
Temperature:75.483,Time:274659
Temperature:75.483,Time:274660
Temperature:75.483,Time:274660
Temperature:76.54,Time:274660
Temperature:76.54,Time:274660
Temperature:76.54,Time:274661
Temperature:76.54,Time:274661
Temperature:75.483,Time:274661

In [8]:
ble.stop_notify(ble.uuid['RX_STRING'])

## Disconnect

In [9]:
# Disconnect
ble.disconnect()